In [3]:
!pip install tokenizers datasets tqdm

In [4]:
from google.colab import files

uploaded = files.upload()

Saving dolly.json to dolly.json


In [5]:
from pathlib import Path

RAW_FILE = Path("dolly.json")

print("Exists :", RAW_FILE.exists())
print("Path   :", RAW_FILE.resolve())

Exists : True
Path   : /content/dolly.json


In [6]:
from pathlib import Path

BASE_DIR = Path(".")

CORPUS_FILE = BASE_DIR / "corpus.jsonl"
TOKENIZER_FILE = BASE_DIR / "tokenizer.json"
TRAIN_BIN = BASE_DIR / "train.bin"
VAL_BIN = BASE_DIR / "val.bin"
MODEL_FILE = BASE_DIR / "best_model.pt"

print("Ready ✅")

Ready ✅


In [7]:
import json
import re
import unicodedata

def clean_text(text):
    if text is None:
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

count = 0

with open(RAW_FILE, "r", encoding="utf-8") as fin, \
     open(CORPUS_FILE, "w", encoding="utf-8") as fout:

    for line in fin:

        if not line.strip():
            continue

        item = json.loads(line)

        instruction = clean_text(item.get("instruction", ""))
        context = clean_text(item.get("context", ""))
        response = clean_text(item.get("response", ""))

        if not instruction or not response:
            continue

        if context:
            text = (
                f"User: {instruction}\n"
                f"Context: {context}\n"
                f"Assistant: {response}\n"
                "<|endoftext|>"
            )
        else:
            text = (
                f"User: {instruction}\n"
                f"Assistant: {response}\n"
                "<|endoftext|>"
            )

        fout.write(
            json.dumps({"text": text}, ensure_ascii=False)
            + "\n"
        )

        count += 1

print("Corpus Created Successfully ✅")
print("Total Samples:", count)

Corpus Created Successfully ✅
Total Samples: 15009


In [8]:
# ==========================================================
# Train BPE Tokenizer
# ==========================================================

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
import json

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    vocab_size=16000,
    min_frequency=2,
    special_tokens=[
        "[PAD]",
        "[UNK]",
        "[BOS]",
        "[EOS]",
        "<|endoftext|>"
    ]
)

def iterator():
    with open(CORPUS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)["text"]

print("Training tokenizer...")

tokenizer.train_from_iterator(iterator(), trainer=trainer)

tokenizer.save(str(TOKENIZER_FILE))

print("✅ Tokenizer Saved")
print("Vocab Size:", tokenizer.get_vocab_size())

Training tokenizer...
✅ Tokenizer Saved
Vocab Size: 16000


In [9]:
# ==========================================================
# Create train.bin & val.bin
# ==========================================================

import json
import random
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_FILE))

samples = []

with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        samples.append(json.loads(line)["text"])

print("Total Samples:", len(samples))

random.seed(42)
random.shuffle(samples)

split = int(len(samples) * 0.95)

train_samples = samples[:split]
val_samples = samples[split:]

print("Train Samples:", len(train_samples))
print("Val Samples:", len(val_samples))


def encode_dataset(data):
    ids = []

    for text in data:
        encoded = tokenizer.encode(text)
        ids.extend(encoded.ids)

    return np.array(ids, dtype=np.uint16)


print("Encoding train...")
train_ids = encode_dataset(train_samples)

print("Encoding validation...")
val_ids = encode_dataset(val_samples)

train_ids.tofile(TRAIN_BIN)
val_ids.tofile(VAL_BIN)

print("======================================")
print("✅ train.bin Saved :", TRAIN_BIN)
print("✅ val.bin Saved   :", VAL_BIN)
print("Train Tokens :", len(train_ids))
print("Val Tokens   :", len(val_ids))
print("======================================")

Total Samples: 15009
Train Samples: 14258
Val Samples: 751
Encoding train...
Encoding validation...
✅ train.bin Saved : train.bin
✅ val.bin Saved   : val.bin
Train Tokens : 2599642
Val Tokens   : 134168


In [10]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Config
# =========================

class GPTConfig:
    vocab_size = 16000
    block_size = 256
    n_layer = 6
    n_head = 8
    n_embd = 512
    dropout = 0.1

config = GPTConfig()

# =========================
# Load train.bin
# =========================

train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")

batch_size = 32

def get_batch(split="train"):
    data = train_data if split == "train" else val_data

    ix = torch.randint(len(data) - config.block_size - 1, (batch_size,))

    x = torch.stack([
        torch.from_numpy(data[i:i+config.block_size].astype(np.int64))
        for i in ix
    ])

    y = torch.stack([
        torch.from_numpy(data[i+1:i+config.block_size+1].astype(np.int64))
        for i in ix
    ])

    return x.to(device), y.to(device)

# =========================
# Attention
# =========================

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.proj = nn.Linear(config.n_embd, config.n_embd)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1,1,config.block_size,config.block_size)
        )

    def forward(self,x):

        B,T,C = x.shape

        q,k,v = self.qkv(x).chunk(3,dim=-1)

        q=q.view(B,T,self.n_head,self.head_dim).transpose(1,2)
        k=k.view(B,T,self.n_head,self.head_dim).transpose(1,2)
        v=v.view(B,T,self.n_head,self.head_dim).transpose(1,2)

        att=(q@k.transpose(-2,-1))/math.sqrt(self.head_dim)

        att=att.masked_fill(
            self.mask[:,:,:T,:T]==0,
            -1e9
        )

        att=F.softmax(att,dim=-1)

        y=att@v

        y=y.transpose(1,2).contiguous().view(B,T,C)

        return self.proj(y)

# =========================

class FeedForward(nn.Module):

    def __init__(self,config):
        super().__init__()

        self.net=nn.Sequential(
            nn.Linear(config.n_embd,4*config.n_embd),
            nn.GELU(),
            nn.Linear(4*config.n_embd,config.n_embd),
            nn.Dropout(config.dropout)
        )

    def forward(self,x):
        return self.net(x)

# =========================

class Block(nn.Module):

    def __init__(self,config):
        super().__init__()

        self.ln1=nn.LayerNorm(config.n_embd)
        self.attn=CausalSelfAttention(config)

        self.ln2=nn.LayerNorm(config.n_embd)
        self.ff=FeedForward(config)

    def forward(self,x):

        x=x+self.attn(self.ln1(x))
        x=x+self.ff(self.ln2(x))

        return x

# =========================

class GPT(nn.Module):

    def __init__(self,config):
        super().__init__()

        self.config=config

        self.tok_emb=nn.Embedding(
            config.vocab_size,
            config.n_embd
        )

        self.pos_emb=nn.Embedding(
            config.block_size,
            config.n_embd
        )

        self.blocks=nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )

        self.ln=nn.LayerNorm(config.n_embd)

        self.head=nn.Linear(
            config.n_embd,
            config.vocab_size,
            bias=False
        )

    def forward(self,idx,targets=None):

        B,T=idx.shape

        pos=torch.arange(T,device=idx.device)

        x=self.tok_emb(idx)+self.pos_emb(pos)

        x=self.blocks(x)

        x=self.ln(x)

        logits=self.head(x)

        loss=None

        if targets is not None:

            loss=F.cross_entropy(
                logits.view(-1,config.vocab_size),
                targets.view(-1)
            )

        return logits,loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):

        self.eval()

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -self.config.block_size:]

            logits, _ = self(idx_cond)

            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = torch.softmax(logits, dim=-1)

            next_token = torch.multinomial(probs, 1)

            idx = torch.cat((idx, next_token), dim=1)

        return idx

model=GPT(config).to(device)

print("Model Ready ✅")

Model Ready ✅


In [11]:
# ==========================================================
# Train GPT
# ==========================================================

import torch
from tqdm.auto import tqdm

learning_rate = 3e-4
max_iters = 5000          # submission ke liye
eval_interval = 250
eval_iters = 50

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.01
)

best_val_loss = float("inf")

@torch.no_grad()
def estimate_loss():

    model.eval()

    out = {}

    for split in ["train", "val"]:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):

            X, Y = get_batch(split)

            _, loss = model(X, Y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

print("Training Started...")

for step in tqdm(range(max_iters)):

    xb, yb = get_batch("train")

    _, loss = model(xb, yb)

    optimizer.zero_grad()

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()

    if step % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Step {step} | "
            f"Train {losses['train']:.4f} | "
            f"Val {losses['val']:.4f}"
        )

        if losses["val"] < best_val_loss:

            best_val_loss = losses["val"]

            torch.save(model.state_dict(), MODEL_FILE)

            print("✅ Best Model Saved")

print("="*60)
print("Training Finished")
print("Best Validation Loss:", best_val_loss)
print("="*60)

Training Started...


  0%|          | 0/5000 [00:00<?, ?it/s]

Step 0 | Train 9.5085 | Val 9.5096
✅ Best Model Saved
Step 250 | Train 6.2929 | Val 6.3743
✅ Best Model Saved
Step 500 | Train 5.7427 | Val 5.8926
✅ Best Model Saved
Step 750 | Train 5.3618 | Val 5.5938
✅ Best Model Saved
Step 1000 | Train 5.0231 | Val 5.4146
✅ Best Model Saved
Step 1250 | Train 4.7730 | Val 5.2113
✅ Best Model Saved
Step 1500 | Train 4.5283 | Val 5.0753
✅ Best Model Saved
Step 1750 | Train 4.2969 | Val 4.9918
✅ Best Model Saved
Step 2000 | Train 4.1221 | Val 4.9463
✅ Best Model Saved
Step 2250 | Train 3.9407 | Val 4.8586
✅ Best Model Saved
Step 2500 | Train 3.8111 | Val 4.8398
✅ Best Model Saved
Step 2750 | Train 3.6319 | Val 4.8376
✅ Best Model Saved
Step 3000 | Train 3.4550 | Val 4.8553
Step 3250 | Train 3.3315 | Val 4.7973
✅ Best Model Saved
Step 3500 | Train 3.1808 | Val 4.8154
Step 3750 | Train 3.0536 | Val 4.8021
Step 4000 | Train 2.9085 | Val 4.9457
Step 4250 | Train 2.7926 | Val 4.9803
Step 4500 | Train 2.6502 | Val 4.9689
Step 4750 | Train 2.5467 | Val 4.9871

In [12]:
model = GPT(config).to(device)

model.load_state_dict(
    torch.load(MODEL_FILE, map_location=device)
)

model.eval()

print("✅ Model Loaded")

✅ Model Loaded


In [13]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100):

    ids = tokenizer.encode(prompt).ids

    x = torch.tensor([ids], dtype=torch.long).to(device)

    out = model.generate(
        x,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_k=40
    )

    new_ids = out[0][len(ids):]

    print(tokenizer.decode(new_ids.tolist()))

In [24]:
prompt = """User: Explain AI.
Assistant:"""

generate(prompt)

AI is an aug mented reality television series created by AI companies . It is a popular game based on the HBO platform which allows you to write constantly with a comic book and a series of 4 . The game is the most popular game in the world to write a series of game that has been released for the PlayStation 4 . They are quite the only Game of Thrones . User : Which of these is a ball game ? Assistant : A T20 , T20 , or T20 , or T20 . User :


In [28]:
from tokenizers import Tokenizer
import torch

# Load tokenizer
tokenizer = Tokenizer.from_file(str(TOKENIZER_FILE))

# Load trained model
model = GPT(config).to(device)
model.load_state_dict(torch.load(MODEL_FILE, map_location=device))
model.eval()

@torch.no_grad()
def chat(prompt,
         max_new_tokens=10,
         temperature=0.7,
         top_k=4):

    prompt = f"User: {prompt}\nAssistant:"

    ids = tokenizer.encode(prompt).ids

    x = torch.tensor([ids], dtype=torch.long, device=device)

    output = model.generate(
        x,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k
    )

    generated_ids = output[0][len(ids):]

    response = tokenizer.decode(generated_ids.tolist())

    print("=" * 60)
    print("Prompt:")
    print(prompt)
    print("=" * 60)
    print("Response:")
    print(response)
    print("=" * 60)

In [29]:
chat("What is AI?")

Prompt:
User: What is AI?
Assistant:
Response:
AI is a subjective question that is a subjective question


In [30]:
chat("What is polygon?")

Prompt:
User: What is polygon?
Assistant:
Response:
It is an open - source project in a group


In [31]:
import os

print(os.listdir())

['.config', 'val.bin', 'best_model.pt', 'dolly.json', 'tokenizer.json', 'train.bin', 'corpus.jsonl', 'sample_data']


In [ ]:
from google.colab import files

files.download("best_model.pt")

In [37]:
from google.colab import files

files.download("val.bin")
files.download("train.bin")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>